# CAMELS-DE-1h

Compile hourly the final CAMELS-DE-1h dataset.  

In [1]:
from pathlib import Path
import gc

from tqdm import tqdm
import pandas as pd
import polars as pl
import geopandas as gpd
import xarray as xr
import numpy as np
import zarr
import numcodecs
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.animation as animation
from IPython.display import HTML

from camels_de1h import Station1h, get_metadata1h

## Folder to save everything

In [ ]:
out_path = Path("../CAMELS-DE-1h")

out_path.mkdir(exist_ok=True)

## IDs for CAMELS-DE-1h

In [3]:
camels_de_path = Path("/home/alexd/datasets/CAMELS_DE_v1_1_0/")
if not camels_de_path.exists():
    raise FileNotFoundError(f"CAMELS-DE not found at {camels_de_path}")

radklim_original_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/radklim_original1998/")
if not radklim_original_path.exists():
    raise FileNotFoundError(f"Radklim data not found at {radklim_original_path}")

radklim_infilled_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/radklim_infilled1998/")
if not radklim_infilled_path.exists():
    raise FileNotFoundError(f"Radklim infilled data not found at {radklim_infilled_path}")

hostrada_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/hostrada2444_camelsde/")
if not hostrada_path.exists():
    raise FileNotFoundError(f"Hostrada data not found at {hostrada_path}")

icond2_det_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/icond2_deterministic2444_camelsde/")
if not icond2_det_path.exists():
    raise FileNotFoundError(f"ICON D2 deterministic data not found at {icond2_det_path}")

icond2_eps_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/icond2_eps_camelsde1997/")
if not icond2_eps_path.exists():
    raise FileNotFoundError(f"ICON D2 EPS data not found at {icond2_eps_path}")

attributes_path = Path("./output_data/static_attributes/")
if not attributes_path.exists():
    raise FileNotFoundError(f"Static attributes not found at {attributes_path}")

## Filter catchments
* area_metadata and coordinates are not NaN -> MERIT Hydro available
* min. 90% in Germany
* max. diff to reported area: 20%
* min. 1 year of discharge data

In [4]:
meta1h = get_metadata1h()

# 1. only stations with area_metdata and coordinates -> MERIT Hydro catchment available
meta1h = meta1h.dropna(subset=["area_metadata", "easting", "northing"])

# 2. minimum 90 % of the catchment area within Germany
meta1h = meta1h[meta1h["area_pct_in_germany"] >= 90]

# 3. maximum 20 % difference between MERIT Hydro area and reported catchment area
meta1h = meta1h[meta1h["diff_to_reported_area"].abs() <= 20]

# 4. minimum 2 years of discharge data
meta1h = meta1h[meta1h["q_years"] >= 1] 

meta1h = meta1h.sort_values("gauge_id").reset_index(drop=True)

ids = meta1h["gauge_id"].tolist()

meta1h

,gauge_id,provider_id,gauge_name,waterbody_name,federal_state,lon,lat,easting,northing,elev_metadata,area_metadata,part_of_camelsp,area_calc,in_camelsde_1d,area_pct_in_germany,q_years,w_years,diff_to_reported_area
0,DE110000,105,Kirchen-Hausen,Donau,Baden-Württemberg,8.679586,47.925507,4.222273e+06,2.757761e+06,657.334,758.528,True,763.066361,True,100.0,24.00,24.0,0.598312
1,DE110010,106,Möhringen,Donau,Baden-Württemberg,8.760112,47.948961,4.228336e+06,2.760263e+06,649.162,826.963,True,831.294102,True,100.0,23.16,24.0,0.523736
2,DE110020,120,Hundersingen,Donau,Baden-Württemberg,9.396100,48.072462,4.275974e+06,2.773394e+06,542.530,2621.324,True,2618.597245,True,100.0,24.00,24.0,-0.104022
3,DE110030,125,Berg,Donau,Baden-Württemberg,9.731457,48.266270,4.301054e+06,2.794786e+06,489.903,4072.790,True,4093.963291,True,100.0,24.00,24.0,0.519872
4,DE110040,129,Achstetten,Baierzer Rot,Baden-Württemberg,9.900817,48.263440,4.313633e+06,2.794441e+06,489.317,264.393,True,272.811527,True,100.0,24.00,24.0,3.184096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1632,DEG11340,577310,Staitz (AP),Weida,Thüringen,12.002473,50.707509,4.462436e+06,3.068139e+06,287.191,165.000,False,166.177402,False,100.0,12.00,0.0,0.713577
1633,DEG11350,577340,Hohenleuben (ZP),Leuba,Thüringen,12.076081,50.703843,4.467645e+06,3.067877e+06,305.536,28.200,False,28.552454,False,100.0,11.92,0.0,1.249837
1634,DEG11360,577342,Hohenleuben (AP),Leuba,Thüringen,12.068883,50.724615,4.467072e+06,3.070172e+06,279.136,41.400,False,42.110346,False,100.0,8.84,0.0,1.715811
1635,DEG11380,577350,Weida-Heinoldsmühle,Auma,Thüringen,12.015941,50.768096,4.463203e+06,3.074903e+06,271.878,106.900,False,106.789596,False,100.0,12.00,0.0,0.103277


In [5]:
# check if meteorological data is available for all stations in filtered meta1h
ids_radklim_original = sorted([p.name.split("_")[0] for p in radklim_original_path.glob("DE*.csv")])
ids_radklim_infilled = sorted([p.name.split("_")[0] for p in radklim_infilled_path.glob("DE*.csv")])
ids_hostrada = sorted([p.name.split("/")[-1].split("_")[0] for p in set(hostrada_path.glob("*.csv"))])
ids_icond2_det = sorted([p.name.split("_")[3] for p in icond2_det_path.glob("*.nc")])
ids_icond2_eps = sorted([p.name.split("_")[-1].split(".")[0] for p in icond2_eps_path.glob("*.nc")])

# ids merit hydro catchments
gdf_catchments = gpd.read_file("./output_data/merit_hydro/CAMELS_DE_1h_catchments.gpkg")
ids_catchments = sorted(gdf_catchments["gauge_id"].tolist())

# check if all ids have data in all datasets
for dataset_name, dataset_ids in zip(
    ["CATCHMENTS", "Radklim Original", "Radklim Infilled", "Hostrada", "ICON D2 Deterministic", "ICON D2 EPS"],
    [ids_catchments, ids_radklim_original, ids_radklim_infilled, ids_hostrada, ids_icond2_det, ids_icond2_eps],
):
    missing_ids = set(ids) - set(dataset_ids)
    if missing_ids:
        print(f"Warning: The following IDs are missing in {dataset_name} dataset: {sorted(missing_ids)}")
    else:
        print(f"All IDs are present in {dataset_name} dataset.")


All IDs are present in CATCHMENTS dataset.
All IDs are present in Radklim Original dataset.
All IDs are present in Radklim Infilled dataset.
All IDs are present in Hostrada dataset.
All IDs are present in ICON D2 Deterministic dataset.
All IDs are present in ICON D2 EPS dataset.


# QC: remove some individual catchments

* DEA13180: many (highly) negative discharge values, suspicious timeseries (maybe okay / no errors), name "Sülzüberleitung" indicates artificial flow
* DEF14100: very weird looking hydrograph, probably errors in discharge data
* DEF14500: not enough data to calculate hydrologic and climatic attributes (no complete hydrologic year)
* DED10560: many gaps in the discharge time series, no complete hydrologic year, no climatic attributes
* DE210390: "Illerkanal": highly artificial flow, issues in timeseries with long windows of drops in the discharge time series
* Erftverband: Pegel Bedburg und Schwerfen (errors in data / uncertain data quality)
* remove a total of 19 catchments where precipitation data is missing for multiple years (or no "original" data at all), as the areas were not always covered by a radar device (most southern part of Bavaria and north-western part of Niedersachsen). If it was only partial gaps with a maximum of 10 percent missing data in the gap, the catchments were kept, as the data also looks good from visual inspection

additionally, all negative discharge values in all catchments were replaced with NaN

In [6]:
print("IDs before exclusions:", len(ids))

# individual issues
ids_exclude = ["DEA13180", "DEF14100", "DEF14500", "DED10560", "DE210390"]

# remove IDs with partial Radklim time series and high gap percentage? -> Niedersachsen & Bayern
ids_exclude_long_gaps = ["DE212670", "DE213640", "DE212730", "DE212760", "DE213630", "DE213390", "DE213400", "DE215370", "DE215380", "DE210250", "DE210210", "DE210220", "DE210230", "DE210260", "DE210290", "DE910570", "DE910580", "DE910600", "DE912150"]

# Remove 2 Erftverband gauges -> Pegel Bedburg und Schwerfen (errors in data / uncertain data quality)
ids_erft_exclude = ["DEA13620", "DEA13740"]

ids = [id for id in ids if id not in ids_exclude]
ids = [id for id in ids if id not in ids_exclude_long_gaps]
ids = [id for id in ids if id not in ids_erft_exclude]

print("IDs after exclusions:", len(ids))

IDs before exclusions: 1637
IDs after exclusions: 1611


## Attributes and geopackage files


In [7]:
attribute_files = attributes_path.glob("CAMELS_DE_1h*attributes.csv")
list(attribute_files)

[PosixPath('output_data/static_attributes/CAMELS_DE_1h_landcover_attributes.csv'),
 PosixPath('output_data/static_attributes/CAMELS_DE_1h_humaninfluence_attributes.csv'),
 PosixPath('output_data/static_attributes/CAMELS_DE_1h_soil_attributes.csv'),
 PosixPath('output_data/static_attributes/CAMELS_DE_1h_hydrogeology_attributes.csv'),
 PosixPath('output_data/static_attributes/CAMELS_DE_1h_topographic_attributes.csv')]

In [8]:
# attributes
attribute_files = attributes_path.glob("CAMELS_DE_1h*attributes.csv")

attributes = {}

for attribute_file in attribute_files:
    # read attribute file
    df_attribute = pd.read_csv(attribute_file)

    # filter gauge_id
    df_attribute = df_attribute[df_attribute["gauge_id"].isin(ids)]

    # check that all IDs are there
    assert len(df_attribute) == len(ids)

    if attribute_file.stem == "CAMELS_DE_1h_topographic_attributes":
        # add in_camelsde_1d and area_pct_in_germany from meta1h
        df_attribute = df_attribute.merge(meta1h[["gauge_id", "in_camelsde_1d", "area_pct_in_germany"]], on="gauge_id", how="left")
        # reorder columns
        df_attribute = df_attribute[['gauge_id', 'provider_id', 'gauge_name', 'water_body_name', 'federal_state', 'area_pct_in_germany', 'in_camelsde_1d', 'gauge_lat', 'gauge_lon', 'gauge_easting', 'gauge_northing', 'gauge_elev_metadata', 'area_metadata', 'gauge_elev', 'area', 'elev_mean', 'elev_min', 'elev_5', 'elev_50', 'elev_95', 'elev_max']]

    # add to dict
    attributes[attribute_file.stem] = df_attribute

    # save to out_path
    df_attribute.to_csv(out_path / attribute_file.name, index=False)

In [9]:
# geopackages and shapefiles
gdf_stations = gpd.read_file("./output_data/merit_hydro/CAMELS_DE_1h_gauging_stations.gpkg")
gdf_catchments = gpd.read_file("./output_data/merit_hydro/CAMELS_DE_1h_catchments.gpkg")

# filter gauge_id
gdf_stations = gdf_stations[gdf_stations["gauge_id"].isin(ids)]
gdf_catchments = gdf_catchments[gdf_catchments["gauge_id"].isin(ids)]

# make directories
(out_path / "CAMELS_DE_1h_catchment_boundaries" / "gauging_stations").mkdir(parents=True, exist_ok=True)
(out_path / "CAMELS_DE_1h_catchment_boundaries" / "catchments").mkdir(parents=True, exist_ok=True)

# save to out_path
gdf_stations.to_file(out_path / "CAMELS_DE_1h_catchment_boundaries" / "gauging_stations" / "CAMELS_DE_1h_gauging_stations.gpkg", driver="GPKG")
gdf_stations.to_file(out_path / "CAMELS_DE_1h_catchment_boundaries" / "gauging_stations" / "CAMELS_DE_1h_gauging_stations.shp")

gdf_catchments.to_file(out_path / "CAMELS_DE_1h_catchment_boundaries" / "catchments" / "CAMELS_DE_1h_catchments.gpkg", driver="GPKG")
gdf_catchments.to_file(out_path / "CAMELS_DE_1h_catchment_boundaries" / "catchments" / "CAMELS_DE_1h_catchments.shp")

/tmp/ipykernel_891620/1310174392.py:18: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_catchments.to_file(out_path / "CAMELS_DE_1h_catchment_boundaries" / "catchments" / "CAMELS_DE_1h_catchments.shp")


## Add timeseries data

* discharge and water level + calculate specific discharge
* RADKLIM-YW (DWD): precipitation (original and infilled)
* Hostrada (DWD): cloud cover, wind speed, wind direction, air temperature mean, dew point temperature, relative humidity, water vapor mixing ratio, air pressure sea level, air pressure surface, global shortwave radiation

In [10]:
def read_hydrologic_data(id: str, topographic_attributes: pd.DataFrame) -> pl.DataFrame:
    """
    Read the hydrologic data for a given id.  
    Also calculate the specific discharge.  
    Also replaces all negative discharge values with NaN.

    Returns
    -------
    df : pl.DataFrame
    """
    # read data (assuming Station1h returns a pandas DataFrame, convert to Polars)
    df_pandas = Station1h(id).get_data()
    df = pl.from_pandas(df_pandas)

    # make sure that columns are numeric
    df = df.with_columns([
        pl.col("discharge_vol_obs").cast(pl.Float64),
        pl.col("water_level_obs").cast(pl.Float64)
    ])

    # convert topographic attributes to Polars DataFrame
    topographic_attributes = pl.from_pandas(topographic_attributes)

    # get area of the catchment in m2
    area = (topographic_attributes
            .filter(pl.col("gauge_id") == id)
            .select("area")
            .item() * 1e6)  # in m2

    # calculate specific discharge for each hour
    df = df.with_columns([
        (((pl.col("discharge_vol_obs") * 3600) / area) * 1000).alias("discharge_spec_obs")  # in mm/day
    ])

    # rename column date to time
    df = df.rename({"date": "time"})

    # drop timezone information if timezone exists
    df = df.with_columns([
        pl.col("time").dt.replace_time_zone(None).alias("time")
    ])

    # reorder and select columns
    df = df.select(["time", "discharge_vol_obs", "discharge_spec_obs", "water_level_obs"])

    # round the values to 5 decimal places
    df = df.with_columns([
        pl.col("discharge_vol_obs").round(5),
        pl.col("discharge_spec_obs").round(5),
        pl.col("water_level_obs").round(5)
    ])

    # Make sure datetime precision is microseconds
    df = df.with_columns([
        pl.col("time").cast(pl.Datetime("us")).alias("time")
    ])

    # Replace all negative discharge values with NaN
    df = df.with_columns([
        pl.when(pl.col("discharge_vol_obs") < 0)
        .then(None)
        .otherwise(pl.col("discharge_vol_obs"))
        .alias("discharge_vol_obs"),

        pl.when(pl.col("discharge_spec_obs") < 0)
        .then(None)
        .otherwise(pl.col("discharge_spec_obs"))
        .alias("discharge_spec_obs"),
    ])

    return df


In [11]:
def read_radklim_data(radklim_path: str, id: str) -> pl.DataFrame:
    """
    Reads the radklim data for a given station id and returns as a dataframe.  
    Uses Polars for improved performance.
    """
    # read the data
    try:
        file_path = radklim_path / f"{id}_aggregated.csv"
        radklim = pl.read_csv(file_path, try_parse_dates=True)
    except FileNotFoundError:
        raise FileNotFoundError(f"File {id}_aggregated.csv not found in {radklim_path / 'precipitation' / id}")

    # check for duplicated times
    if radklim.select(pl.col("time")).is_duplicated().any():
        raise ValueError(f"Duplicated times found in RADKLIM data for station {id}")

    # rename the columns from short name to long name
    old_names = radklim.columns
    new_names = {col: col.replace("pr", "precipitation") for col in old_names}
    radklim = radklim.rename(new_names)

    # rename quantile columns: (q=0.1) -> 10, (1=0.2) -> 20, ..., (q=0.8) -> 80, (q=0.9) -> 90
    # rename_dict = {
    #     'RR_quantile10': 'precipitation_sum_quantile10',
    #     'RR_quantile20': 'precipitation_sum_quantile20',
    #     'RR_quantile30': 'precipitation_sum_quantile30',
    #     'RR_quantile40': 'precipitation_sum_quantile40',
    #     'RR_quantile50': 'precipitation_sum_quantile50',
    #     'RR_quantile60': 'precipitation_sum_quantile60',
    #     'RR_quantile70': 'precipitation_sum_quantile70',
    #     'RR_quantile80': 'precipitation_sum_quantile80',
    #     'RR_quantile90': 'precipitation_sum_quantile90'
    # }

    # Rename the columns
    # radklim = radklim.rename(rename_dict)

    # drop quantile columns
    radklim = radklim.drop([col for col in radklim.columns if "quantile" in col])

    # make sure all columns are numeric
    radklim = radklim.with_columns([
        pl.col(col).cast(pl.Float64) for col in radklim.columns if col != "time"
    ])

    # round the values to 2 decimal places
    radklim = radklim.with_columns([
        pl.col(col).round(2) for col in radklim.columns if col != "time"
    ])

    # Make sure datetime precision is microseconds
    radklim = radklim.with_columns([
        pl.col("time").cast(pl.Datetime("us")).alias("time")
    ])

    return radklim

In [12]:
def merge_radklim_original_infilled(df_radklim_original: pl.DataFrame, df_radklim_infilled: pl.DataFrame, mode: str = "gapfilled_columns") -> pl.DataFrame:
    """
    Merge original and gap-filled RADKLIM precipitation data.

    The original RADKLIM data can have gaps when individual radars are temporarily
    out of service. This can lead to catchments that are temporarily only partially covered by
    radar precipitation data or are not covered at all.

    Parameters
    ----------
    mode : str
        "gapfilled_flag" : Single set of precipitation columns where gaps are replaced by the
            infilled version, plus a boolean flag column ``flag_precipitation_gap_filled``.
        "gapfilled_columns" : Original precipitation columns are set to NaN wherever
            ``perc_nan > 0`` (partial or full radar gaps), and additional ``*_gapfilled``
            columns are added that use the infilled values to fill those gaps.
            No flag column; use ``perc_nan_precipitation_original`` instead.

    References
    ----------
    Infilled RADKLIM: https://doi.org/10.5194/hess-29-3687-2025
    """
    precipitation_cols = ["precipitation_min", "precipitation_mean", "precipitation_max",
                          "precipitation_stdev"]

    # Add suffix to distinguish columns
    df_radklim_original_tmp = df_radklim_original.rename({
        col: f"{col}_original" for col in df_radklim_original.columns if col != "time"
    })
    df_radklim_infilled_tmp = df_radklim_infilled.rename({
        col: f"{col}_infilled" for col in df_radklim_infilled.columns if col != "time"
    })

    # Join the dataframes on time
    df_merged = df_radklim_original_tmp.join(df_radklim_infilled_tmp, on="time", how="inner")

    if mode == "gapfilled_flag":
        # Replace precipitation columns with infilled data when perc_nan > 0, add flag column
        df_radklim = df_merged.with_columns([
            (pl.col("perc_nan_original") > 0).alias("flag_precipitation_gap_filled")
        ])
        for col in precipitation_cols:
            df_radklim = df_radklim.with_columns([
                pl.when(pl.col("flag_precipitation_gap_filled"))
                .then(pl.col(f"{col}_infilled"))
                .otherwise(pl.col(f"{col}_original"))
                .alias(col)
            ])
        df_radklim = df_radklim.with_columns([
            pl.col("perc_nan_original").alias("perc_nan_precipitation_original")
        ])
        df_radklim = df_radklim.select([
            "time",
            *precipitation_cols,
            "perc_nan_precipitation_original",
            "flag_precipitation_gap_filled",
        ])

    elif mode == "gapfilled_columns":
        # Original columns: set to NaN wherever perc_nan > 0 (partial or full radar gaps)
        # _gapfilled twins: substitute infilled values for those same timestamps
        original_exprs = [
            pl.when(pl.col("perc_nan_original") > 0)
            .then(None)
            .otherwise(pl.col(f"{col}_original"))
            .alias(col)
            for col in precipitation_cols
        ]
        gapfilled_exprs = [
            pl.when(pl.col("perc_nan_original") > 0)
            .then(pl.col(f"{col}_infilled"))
            .otherwise(pl.col(f"{col}_original"))
            .alias(f"{col}_gapfilled")
            for col in precipitation_cols
        ]

        df_radklim = df_merged.with_columns(original_exprs + gapfilled_exprs + [
            pl.col("perc_nan_original").alias("perc_nan_precipitation_original")
        ])
        df_radklim = df_radklim.select([
            "time",
            *precipitation_cols,
            *[f"{col}_gapfilled" for col in precipitation_cols],
            "perc_nan_precipitation_original",
        ])

    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'gapfilled_flag' or 'gapfilled_columns'.")

    return df_radklim

In [13]:
def read_hostrada_data(hostrada_path: str, id: str) -> pl.DataFrame:
    """
    Reads the hostrada data for a given station id and returns as a dataframe.
    Uses Polars for improved performance.
    """
    variable_mapping = {
        "air_temperature": "tas",
        "relative_humidity": "hurs",
        "water_vapor_mixing_ratio": "mixr",
        "global_radiation": "rsds",
        "air_pressure_sea_level": "psl",
        "air_pressure_surface": "ps",
        "cloud_cover": "clt",
        "wind_direction": "sfcWind_direction",
        "dew_point_temperature": "tdew",
        "urban_heat_island_intensity": "uhi",
        "wind_speed_eastward_mean": "wind_speed_eastward_mean",
        "wind_speed_northward_mean": "wind_speed_northward_mean",
        "wind_speed_mean": "wind_speed_mean",
        "wind_direction_mean": "wind_direction_mean"
    }

    # Read the data once
    file_path = hostrada_path / f"{id}_hostrada_camelsde.csv"
    data = pl.read_csv(file_path, try_parse_dates=True)

    # check for duplicated times
    if data.select(pl.col("time")).is_duplicated().any():
        # Get rows with duplicated times
        duplicated_mask = data.select(pl.col("time")).is_duplicated()
        duplicated_times = data.filter(duplicated_mask)["time"].unique()
        
        # Check if duplicate rows have identical values
        for dup_time in duplicated_times:
            dup_rows = data.filter(pl.col("time") == dup_time)
            
            # Check if all rows with this timestamp are identical (excluding time column)
            if dup_rows.select(pl.all().exclude("time")).n_unique() > 1:
                raise ValueError(f"Duplicated times found in HOSTRADA data for station {id} with different values at {dup_time}")
        
        # Remove duplicates (keep first occurrence)
        data = data.unique(subset=["time"], keep="first")
    
    # Rename all columns at once
    new_names = {}
    for long_name, short_name in variable_mapping.items():
        for col in data.columns:
            col_var_name = col.split("_")[0]
            if short_name == col_var_name:
                new_names[col] = col.replace(short_name, long_name)
    
    data = data.rename(new_names)

    # Only select columns where colname includes time, min, mean, max, stdev
    data = data.select([col for col in data.columns if "time" in col or "min" in col or "mean" in col or "max" in col or "stdev" in col])

    # Drop quantile columns
    data = data.select([col for col in data.columns if "quantile" not in col])

    # Drop urban_heat_island_intensity_mean column
    data = data.drop("urban_heat_island_intensity_mean")

    # Convert cloud cover from okta to percentage (0-8 okta -> 0-100 %)
    data = data.with_columns([
        pl.when(pl.col("cloud_cover_mean").is_not_null())
        .then(pl.col("cloud_cover_mean") * 12.5)
        .otherwise(pl.col("cloud_cover_mean"))
        .alias("cloud_cover_mean")
    ])

    # Round all values to 2 decimal places (only numeric columns)
    numeric_cols = [col for col in data.columns if col != "time"]
    data = data.with_columns([
        pl.col(col).round(2) for col in numeric_cols
    ])

    # Make sure datetime precision is microseconds
    data = data.with_columns([
        pl.col("time").cast(pl.Datetime("us")).alias("time")
    ])

    return data

In [14]:
id = "DE110000"

# read radklim data
try:
    df_radklim_original = read_radklim_data(radklim_original_path, id)
except FileNotFoundError:
    print(f"File {id}_aggregated.csv not found in {radklim_original_path}")
try:
    df_radklim_infilled = read_radklim_data(radklim_infilled_path, id)
except FileNotFoundError:
    print(f"File {id}_aggregated.csv not found in {radklim_infilled_path}")

merged_radklim = merge_radklim_original_infilled(df_radklim_original, df_radklim_infilled, mode="gapfilled_flag")
print(f"Number of timestamps replaced with infilled data for station {id}: {merged_radklim.filter(pl.col('flag_precipitation_gap_filled')).shape[0]}")

Number of timestamps replaced with infilled data for station DE110000: 5780


In [15]:
id = "DE212810"

# read radklim data
try:
    df_radklim_original = read_radklim_data(radklim_original_path, id)
except FileNotFoundError:
    print(f"File {id}_aggregated.csv not found in {radklim_original_path}")
try:
    df_radklim_infilled = read_radklim_data(radklim_infilled_path, id)
except FileNotFoundError:
    print(f"File {id}_aggregated.csv not found in {radklim_infilled_path}")

merged_radklim = merge_radklim_original_infilled(df_radklim_original, df_radklim_infilled, mode="gapfilled_flag")
print(f"Number of timestamps replaced with infilled data for station {id}: {merged_radklim.filter(pl.col('flag_precipitation_gap_filled')).shape[0]}")

Number of timestamps replaced with infilled data for station DE212810: 210383


In [16]:
ids_failed = []

for id in tqdm(ids, desc="Processing catchments"):
    # skip if already exists
    if (out_path / "timeseries" / f"CAMELS_DE_1h_hydromet_timeseries_{id}.csv").exists():
        continue

    # just try, if something fails, skip the id
    try:
        # read hydrologic data
        try:
            df_hydrologic = read_hydrologic_data(id, attributes["CAMELS_DE_1h_topographic_attributes"])
        except FileNotFoundError:
            print(f"File {id}_aggregated.csv not found in {camels_de_path}")
            continue

        # read radklim data
        try:
            df_radklim_original = read_radklim_data(radklim_original_path, id)
        except FileNotFoundError:
            print(f"File {id}_aggregated.csv not found in {radklim_original_path}")
            continue
        try:
            df_radklim_infilled = read_radklim_data(radklim_infilled_path, id)
        except FileNotFoundError:
            print(f"File {id}_aggregated.csv not found in {radklim_infilled_path}")
            continue

        # merge original and infilled radklim data
        df_radklim = merge_radklim_original_infilled(df_radklim_original, df_radklim_infilled, mode="gapfilled_columns")

        # read hostrada data
        try:
            df_hostrada = read_hostrada_data(hostrada_path, id)
        except FileNotFoundError:
            print(f"File {id}_aggregated.csv not found in {hostrada_path}")
            continue

        # merge the dataframes using joins, right join on radklim to make sure all dfs are from 2001-01-01 to 2024-01-01
        df = df_radklim.join(df_hydrologic, on="time", how="left")
        df = df.join(df_hostrada, on="time", how="left")

        # put columns discharge_vol_obs,discharge_spec_obs,water_level_obs at the beginning
        cols = df.columns
        cols = [col for col in cols if col not in ["time", "discharge_vol_obs", "discharge_spec_obs", "water_level_obs"]]
        cols = ["time", "discharge_vol_obs", "discharge_spec_obs", "water_level_obs"] + cols
        df = df.select(cols)

        # check for duplicates
        if df["time"].is_duplicated().any():
            duplicate_count = df["time"].is_duplicated().sum()
            raise ValueError(f"Found {duplicate_count} duplicate timestamps.")

        # check for continuity (1-hour steps)
        expected_range = pd.date_range(start=df["time"].min(), end=df["time"].max(), freq='h')
        
        if len(df) != len(expected_range):
            missing = len(expected_range) - len(df)
            # If positive, you have gaps; if negative, you have extra/unexpected rows
            raise ValueError(f"Time continuity break! Expected {len(expected_range)} steps, got {len(df)}. Missing {missing} hours.")
    
        # rename time column to date
        df = df.rename({"time": "date"})

        # make directory to save time series
        (out_path / "timeseries").mkdir(parents=True, exist_ok=True)

        # save to out_path
        output_file = out_path / "timeseries" / f"CAMELS_DE_1h_hydromet_timeseries_{id}.csv"
        df.write_csv(output_file, datetime_format="%Y-%m-%d %H:%M:%S")
    except Exception as e:
        print(f"Error processing {id}: {e}")
        ids_failed.append(id)
        continue
    finally:
        del df_hydrologic, df_radklim, df_hostrada, df  # clean up to free memory
        gc.collect()


Processing catchments:   0%|          | 0/1611 [00:00<?, ?it/s]

Processing catchments: 100%|██████████| 1611/1611 [2:10:08<00:00,  4.85s/it] 


## Add forecast data

The ICON-D2 forecasts data (deterministic and ensemble) were processed on the HPC, which resulted in single .nc files for each catchment.  
These .nc files are converted to .zarr stores and afterwards to zarr ZipStores, which include all catchments.

### ICON-D2 deterministic

In [32]:
icond2_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/icond2_deterministic2444_camelsde/")

# filter forecast files for ids
files = [p for p in icond2_path.glob("*.nc") if any(id in p.name for id in ids)]

print(f"Found {len(files)} ICON D2 deterministic files for {len(ids)} stations.")

# zarr store path (directly in forecasts folder)
forecasts_dir = out_path / "forecasts"
forecasts_dir.mkdir(parents=True, exist_ok=True)
zarr_path = forecasts_dir / "CAMELS_DE_1h_deterministic_met_forecast.zarr"

# SHUFFLE (byte shuffle) is better than BITSHUFFLE for float32 meteorological data
compressor = numcodecs.Blosc(cname='zstd', clevel=6, shuffle=numcodecs.Blosc.SHUFFLE)

zarr_initialized = False

for id in tqdm(ids):
    matching_files = [p for p in files if id in p.name]
    if not matching_files:
        print(f"Warning: No ICON D2 deterministic file found for station ID {id}")
        continue
    if len(matching_files) > 1:
        print(f"Warning: Multiple ICON D2 deterministic files found for station ID {id}: {[p.name for p in matching_files]}")

    # read dataset
    ds = xr.open_dataset(matching_files[0])

    # remove lead_time=0
    ds = ds.sel(lead_time=ds.lead_time != np.timedelta64(0, 'h'))

    # remove attrs["note"] from all precipitation variables
    for var in ds.data_vars:
        if "precipitation" in var:
            if "note" in ds[var].attrs:
                del ds[var].attrs["note"]
            ds[var].attrs["description"] = "Precipitation rate over forecast period"

    # add note to dataset that lead time changes from 27 to 48 hours on 2021-06-23 09:00
    ds.attrs["note"] = "Forecast horizon / max. lead time changes from 27 to 48 hours on init_time 2021-06-23 09:00:00"

    # change the title
    ds.attrs["title"] = f"ICON-D2 deterministic forecast data for CAMELS-DE-1h"

    # rename variable temperature_mean to air_temperature_mean
    ds = ds.rename({"temperature_mean": "air_temperature_mean"})

    # remove init_time '2021-02-10T09:00:00.000000000' to be in line with ensemble forecasts
    ds = ds.sel(init_time=ds.init_time != np.datetime64('2021-02-10T09:00:00.000000000'))

    # remove 2025-01, 2025-02
    ds = ds.sel(init_time=ds.init_time.dt.year < 2025)

    # Convert variables to float32
    for var in ds.data_vars:
        ds[var] = ds[var].astype(np.float32)

    # round to 2 decimal places
    ds = ds.round(2)

    # set precipitation values < 0 (-0.01 can occur, probably due to deaccumulation of cumulative raw precipitation) to 0
    for var in ["precipitation_min", "precipitation_mean", "precipitation_max", "precipitation_stdev"]:
        if ds[var].min() < -0.1:
            print(f"Warning: Negative precipitation values < -0.1 found in station {id} for variable {var}.")
        ds[var] = ds[var].where(~((ds[var] <= 0) & ds[var].notnull()), 0)

    # Add gauge_id as a dimension
    ds = ds.expand_dims(gauge_id=np.array([id], dtype="<U8"))
    ds["gauge_id"].attrs = {
        "description": "CAMELS-DE catchment identifier",
        "long_name": "CAMELS-DE identifier",
    }

    # Verify the result
    times = ds.init_time.to_pandas()
    diffs = times.diff()[1:]
    init_ok = times.is_monotonic_increasing and (diffs == pd.Timedelta(hours=3)).all()
    lead_ok = len(ds.lead_time) == 48 and ds.lead_time.values[0] == pd.Timedelta(hours=1) and ds.lead_time.values[-1] == pd.Timedelta(hours=48)

    # raise Error if checks fail
    if not (init_ok and lead_ok):
        raise ValueError(f"ISSUE in {id}: init_time_ok={init_ok}, lead_time_ok={lead_ok}")

    n_init = len(ds.init_time)
    n_lead = len(ds.lead_time)

    if not zarr_initialized:
        # chunk size 1 along gauge_id, full init_time and lead_time per chunk (optimal for single-station reads)
        encoding = {}
        for var in ds.data_vars:
            encoding[var] = {
                'compressor': compressor,
                'chunks': (1, n_init, n_lead),
            }
        ds.to_zarr(zarr_path, mode='w', encoding=encoding)
        zarr_initialized = True
    else:
        ds.to_zarr(zarr_path, append_dim='gauge_id')

    ds.close()

# Consolidate zarr metadata for fast reads
zarr.consolidate_metadata(str(zarr_path))
print(f"Saved zarr store with all stations to {zarr_path}")

Found 1611 ICON D2 deterministic files for 1611 stations.


100%|██████████| 1611/1611 [20:18<00:00,  1.32it/s]

Saved zarr store with all stations to CAMELS-DE-1h/forecasts/CAMELS_DE_1h_deterministic_met_forecast.zarr


### ICON-D2 EPS (Ensemble)

In [8]:
icond2_eps_path = Path("/mnt/data/stgrid2area_done/CAMELS_DE_1h/icond2_eps_camelsde1997/")

# filter forecast files for ids
files = [p for p in icond2_eps_path.glob("*.nc") if any(id in p.name for id in ids)]

print(f"Found {len(files)} ICON D2 ensemble files for {len(ids)} stations.")

# zarr store path (directly in forecasts folder)
forecasts_dir = out_path / "forecasts"
forecasts_dir.mkdir(parents=True, exist_ok=True)
zarr_path = forecasts_dir / "CAMELS_DE_1h_ensemble_met_forecast.zarr"

# SHUFFLE (byte shuffle) is better than BITSHUFFLE for float32 meteorological data
compressor = numcodecs.Blosc(cname='zstd', clevel=6, shuffle=numcodecs.Blosc.SHUFFLE)

zarr_initialized = False

for id in tqdm(ids):
    matching_files = [p for p in files if id in p.name]
    if not matching_files:
        print(f"Warning: No ICON D2 ensemble file found for station ID {id}")
        continue
    if len(matching_files) > 1:
        print(f"Warning: Multiple ICON D2 ensemble files found for station ID {id}: {[p.name for p in matching_files]}")

    # read dataset
    ds = xr.open_dataset(matching_files[0])

    # add note to dataset that lead time changes from 27 to 48 hours on 2021-06-23 09:00
    ds.attrs["note"] = "Forecast horizon / max. lead time changes from 27 to 48 hours on init_time 2021-06-23 09:00:00"

    # change the title
    ds.attrs["title"] = f"ICON-D2 ensemble forecast data for CAMELS-DE-1h"

    # remove attrs["note"] from all precipitation variables
    for var in ds.data_vars:
        if "precipitation" in var:
            if "note" in ds[var].attrs:
                del ds[var].attrs["note"]
            ds[var].attrs["description"] = "Precipitation rate over forecast period"

    # Convert variables to float32
    for var in ds.data_vars:
        ds[var] = ds[var].astype(np.float32)

    # convert data type of gauge_id to string
    ds["gauge_id"] = ds["gauge_id"].astype(str)

    # round to 2 decimal places
    ds = ds.round(2)

    # set precipitation values < 0 to 0
    for var in ["precipitation_min", "precipitation_mean", "precipitation_max", "precipitation_stdev"]:
        if ds[var].min() < -0.1:
            print(f"Warning: Negative precipitation values < -0.1 found in station {id} for variable {var}.")
        ds[var] = ds[var].where(~((ds[var] <= 0) & ds[var].notnull()), 0)

    # Verify the result
    times = ds.init_time.to_pandas()
    diffs = times.diff()[1:]
    init_ok = times.is_monotonic_increasing and (diffs == pd.Timedelta(hours=3)).all()
    lead_ok = len(ds.lead_time) == 48 and ds.lead_time.values[0] == pd.Timedelta(hours=1) and ds.lead_time.values[-1] == pd.Timedelta(hours=48)

    # raise Error if checks fail
    if not (init_ok and lead_ok):
        raise ValueError(f"ISSUE in {id}: init_time_ok={init_ok}, lead_time_ok={lead_ok}")

    n_init = len(ds.init_time)
    n_lead = len(ds.lead_time)
    n_members = len(ds.ensemble_member)

    if not zarr_initialized:
        # chunk size 1 along gauge_id, all init_times, lead_times and ensemble_members per chunk
        encoding = {}
        for var in ds.data_vars:
            encoding[var] = {
                'compressor': compressor,
                'chunks': (1, n_init, n_lead, n_members),
            }
        ds.to_zarr(zarr_path, mode='w', encoding=encoding)
        zarr_initialized = True
    else:
        ds.to_zarr(zarr_path, append_dim='gauge_id')

    ds.close()

# Consolidate zarr metadata for fast reads
zarr.consolidate_metadata(str(zarr_path))
print(f"Saved zarr store with all stations to {zarr_path}")


Found 1611 ICON D2 ensemble files for 1611 stations.


100%|██████████| 1611/1611 [1:52:42<00:00,  4.20s/it]

Saved zarr store with all stations to CAMELS-DE-1h/forecasts/CAMELS_DE_1h_ensemble_met_forecast.zarr


Create zarr ZipStores for the deterministic and ensemble forecasts.

In [ ]:
import zarr
import zipfile
import numcodecs

input_path = out_path / 'forecasts/CAMELS_DE_1h_deterministic_met_forecast.zarr/'
output_path = out_path / 'CAMELS_DE_1h_deterministic_met_forecast.zarr.zip'

source_group = zarr.open(input_path)
char_codec = numcodecs.VLenUTF8()

def is_string_array(item):
    """Check if a Zarr object array actually contains strings."""
    if item.dtype != object or item.size == 0:
        return False
    # Load just the first element by building a (0,) index for each dimension
    first = item[tuple(0 for _ in item.shape)]
    return isinstance(first, str)

def copy_group(source, dest):
    for key in source.keys():
        item = source[key]
        
        if isinstance(item, zarr.Group):
            sub = dest.require_group(key)
            sub.attrs.update(item.attrs)
            copy_group(item, sub)
        
        elif isinstance(item, zarr.Array):
            kws = dict(
                name=key,
                data=item[:],
                chunks=item.chunks,
                compressor=item.compressor,
                fill_value=item.fill_value,
                dtype=item.dtype,
            )
            if is_string_array(item):
                kws['object_codec'] = char_codec
            
            dest.array(**kws)
            dest[key].attrs.update(item.attrs)
            print(f"copy {dest.name}/{key} {item.shape} {item.dtype}")

print(f"Copying {input_path} to {output_path}...")

with zarr.ZipStore(output_path, mode='w', compression=zipfile.ZIP_STORED) as target_store:
    target_group = zarr.group(target_store)
    target_group.attrs.update(source_group.attrs)
    copy_group(source_group, target_group)
    zarr.consolidate_metadata(target_store)

print("Archive created successfully.")

Copying ./CAMELS-DE-1h/forecasts/CAMELS_DE_1h_deterministic_met_forecast.zarr/ to /mnt/data/CAMELS_DE_1h_deterministic_met_forecast.zarr.zip...
copy //air_pressure_sea_level_mean (1611, 11364, 48) float32
copy //air_temperature_mean (1611, 11364, 48) float32
copy //gauge_id (1611,) object
copy //global_radiation_mean (1611, 11364, 48) float32
copy //init_time (11364,) int32
copy //lead_time (48,) int32
copy //precipitation_max (1611, 11364, 48) float32
copy //precipitation_mean (1611, 11364, 48) float32
copy //precipitation_min (1611, 11364, 48) float32
copy //precipitation_stdev (1611, 11364, 48) float32
copy //snow_depth_water_eq_mean (1611, 11364, 48) float32
copy //wind_direction_mean (1611, 11364, 48) float32
copy //wind_speed_eastward_mean (1611, 11364, 48) float32
copy //wind_speed_mean (1611, 11364, 48) float32
copy //wind_speed_northward_mean (1611, 11364, 48) float32
Archive created successfully.


In [ ]:
import zarr
import zipfile
import numcodecs
import itertools

input_path = './CAMELS-DE-1h/forecasts/CAMELS_DE_1h_ensemble_met_forecast.zarr/'
output_path = './CAMELS-DE-1h/CAMELS_DE_1h_ensemble_met_forecast.zarr.zip'

source_group = zarr.open(input_path)
char_codec = numcodecs.VLenUTF8()

def is_string_array(item):
    if item.dtype != object or item.size == 0:
        return False
    first = item[tuple(0 for _ in item.shape)]
    return isinstance(first, str)

def copy_array_chunked(source_arr, dest_arr):
    chunks = source_arr.chunks
    shape = source_arr.shape
    chunk_offsets = [range(0, s, c) for s, c in zip(shape, chunks)]
    for offset in itertools.product(*chunk_offsets):
        sel = tuple(slice(o, min(s, o + c)) for o, s, c in zip(offset, shape, chunks))
        dest_arr[sel] = source_arr[sel]

def copy_group(source, dest):
    for key in source.keys():
        item = source[key]
        
        if isinstance(item, zarr.Group):
            sub = dest.require_group(key)
            sub.attrs.update(item.attrs)
            copy_group(item, sub)
        
        elif isinstance(item, zarr.Array):
            kws = dict(
                name=key,
                shape=item.shape,
                chunks=item.chunks,
                dtype=item.dtype,
                compressor=item.compressor,
                fill_value=item.fill_value,  # passed to create(), not empty()
            )
            if is_string_array(item):
                kws['object_codec'] = char_codec

            dest_arr = dest.full(**kws)  # use full() instead — it respects fill_value
            copy_array_chunked(item, dest_arr)
            dest_arr.attrs.update(item.attrs)
            print(f"copy {dest.name}/{key} {item.shape} {item.dtype}")

print(f"Copying {input_path} to {output_path}...")

with zarr.ZipStore(output_path, mode='w', compression=zipfile.ZIP_STORED) as target_store:
    target_group = zarr.group(target_store)
    target_group.attrs.update(source_group.attrs)
    copy_group(source_group, target_group)
    zarr.consolidate_metadata(target_store)

print("Archive created successfully.")

Copying ./CAMELS-DE-1h/forecasts/CAMELS_DE_1h_ensemble_met_forecast.zarr/ to /mnt/data/CAMELS_DE_1h_ensemble_met_forecast.zarr.zip...
copy //ensemble_member (20,) int32
copy //gauge_id (1611,) <U8
copy //init_time (11364,) int32
copy //lead_time (48,) int32
copy //precipitation_max (1611, 11364, 20, 48) float32
copy //precipitation_mean (1611, 11364, 20, 48) float32
copy //precipitation_min (1611, 11364, 20, 48) float32
copy //precipitation_stdev (1611, 11364, 20, 48) float32
Archive created successfully.
